In [1]:
import polars as pl
import numpy as np
from sklearn.linear_model import BayesianRidge
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
import os
import warnings
warnings.filterwarnings('ignore')

import random
import joblib
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)
seed_everything(seed=2024)

In [2]:
def custom_metric(y_true,y_pred,weight):
    weighted_r2=1-(np.sum(weight*(y_true-y_pred)**2)/np.sum(weight*y_true**2))
    return weighted_r2

train=pl.scan_parquet("./kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/**/*.parquet").collect().to_pandas()
# train = train.sort_values(by=['date_id', 'time_id'])
# train = train.groupby('symbol_id').apply(lambda group: group.ffill().bfill()).reset_index(drop=True)
train.dropna(inplace=True)
print("Length of train data: ", len(train))
# shuffle
# train = train.sample(frac=1).reset_index(drop=True)
# reset index
train.reset_index(drop=True, inplace=True)

cols=[f'feature_0{i}' if i<10 else f'feature_{i}' for i in range(79)]
X=train[cols].values
y=train['responder_6'].values
weights = train['weight'].values
print("train test split")

# 5 fold cv
n_folds = 3
splits = {
    0: {"train_index": 11000000, "test_index": 14000000},
    1: {"train_index": 22000000, "test_index": 25000000},
    2: {"train_index": 33000000, "test_index": -1},
}


# # Use only the last 10 million rows
# split1 = 15000000
# # Use the last 2 million rows of the 10 million for validation
# split2 = 3000000

# train_X = X[-split1:-split2]
# train_y = y[-split1:-split2]
# train_weight = weights[-split1:-split2]

# test_X = X[-split2:]
# test_y = y[-split2:]
# test_weight = weights[-split2:]

# print(f"train_X.shape: {train_X.shape}, test_X.shape: {test_X.shape}")
# print("fit and predict")

Length of train data:  35370822
train test split


In [3]:
import optuna
del train

In [4]:
best_params = []
# scaler = MinMaxScaler()

for fold in range(len(splits)):
    print(f"Fold: {fold}")
    if fold == 0:
        X_train = X[:11000000]
        y_train = y[:11000000]
        weight_train = weights[:11000000]
        X_test = X[11000000:]
        y_test = y[11000000:]
        weight_test = weights[11000000:]
    elif fold == 1:
        X_train = X[11000000:22000000]
        y_train = y[11000000:22000000]
        weight_train = weights[11000000:22000000]
        X_test = np.concatenate((X[:11000000], X[22000000:]), axis=0)
        y_test = np.concatenate((y[:11000000], y[22000000:]), axis=0)
        weight_test = np.concatenate((weights[:11000000], weights[22000000:]), axis=0)
    else:
        X_train = X[22000000:]
        y_train = y[22000000:]
        weight_train = weights[22000000:]
        X_test = X[:22000000]
        y_test = y[:22000000]
        weight_test = weights[:22000000]

    # X_train = scaler.fit_transform(X_train)
    # X_test = scaler.transform(X_test)
    # joblib.dump(scaler, f"scaler_{fold}.pkl")

    def objective(trial):
        params = {
            "alpha_1": trial.suggest_float("alpha_1", 1e-10, 10, log=True),
            "alpha_2": trial.suggest_float("alpha_2", 1e-10, 10, log=True),
            "lambda_1": trial.suggest_float("lambda_1", 1e-10, 10, log=True),
            "lambda_2": trial.suggest_float("lambda_2", 1e-10, 10, log=True),
            "max_iter": trial.suggest_int("max_iter", 100, 1000),
            "tol": trial.suggest_float("tol", 1e-5, 1, log=True),
            "fit_intercept": True,
        }

        model = BayesianRidge(**params)
        model.fit(X_train, y_train, sample_weight=weight_train)
        y_pred = model.predict(X_test)
        score = custom_metric(y_test, y_pred, weight_test)
        return score

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=200, gc_after_trial=True)

    print("Best trial:")
    print("  Value: ", study.best_trial.value)
    print("  Params: ")
    for key, value in study.best_trial.params.items():
        print(f"    {key}: {value}")
    best_params.append(study.best_trial.params)

    best_model = BayesianRidge(**study.best_trial.params)
    best_model.fit(X_train, y_train, sample_weight=weight_train)
    joblib.dump(best_model, f"model_{fold}.pkl")

[I 2024-11-20 20:20:59,885] A new study created in memory with name: no-name-177696e2-f00a-4fd9-b1e0-195ab4fc1c30


Fold: 0


[I 2024-11-20 20:21:13,339] Trial 0 finished with value: 0.002926349639892578 and parameters: {'alpha_1': 6.610086583781077e-07, 'alpha_2': 3.091033883491849e-09, 'lambda_1': 0.00019095359915750142, 'lambda_2': 0.09581542568791938, 'max_iter': 899, 'tol': 9.830723801089961e-05}. Best is trial 0 with value: 0.002926349639892578.
[I 2024-11-20 20:21:26,005] Trial 1 finished with value: 0.0029466748237609863 and parameters: {'alpha_1': 0.0003609589438238722, 'alpha_2': 3.3925084213409072e-09, 'lambda_1': 7.71400534706392e-08, 'lambda_2': 4.4072833649717225e-05, 'max_iter': 322, 'tol': 0.007800663441621025}. Best is trial 1 with value: 0.0029466748237609863.
[I 2024-11-20 20:21:39,200] Trial 2 finished with value: 0.0029425621032714844 and parameters: {'alpha_1': 0.8926959414131186, 'alpha_2': 8.801958055799914e-07, 'lambda_1': 2.6731383795756346e-06, 'lambda_2': 0.0015757964440178212, 'max_iter': 384, 'tol': 0.020099726960112273}. Best is trial 1 with value: 0.0029466748237609863.
[I 2024

Best trial:
  Value:  0.002953171730041504
  Params: 
    alpha_1: 4.077322058661726e-07
    alpha_2: 1.9720585816758267
    lambda_1: 9.985529394697059
    lambda_2: 3.6651819537095414e-06
    max_iter: 837
    tol: 0.0001612709182573927
Fold: 1


[I 2024-11-20 21:02:31,443] A new study created in memory with name: no-name-b7e33689-080b-4e86-a00d-55648ac1e8cc
[I 2024-11-20 21:02:44,444] Trial 0 finished with value: 0.005820631980895996 and parameters: {'alpha_1': 1.2245901318359747e-10, 'alpha_2': 1.5511264146946406e-08, 'lambda_1': 2.4967245167712864e-08, 'lambda_2': 1.5763286066898475e-10, 'max_iter': 735, 'tol': 0.07335350952392526}. Best is trial 0 with value: 0.005820631980895996.
[I 2024-11-20 21:02:56,672] Trial 1 finished with value: 0.005820631980895996 and parameters: {'alpha_1': 0.0003344989627152904, 'alpha_2': 3.7664396613331813e-10, 'lambda_1': 4.1421351770305e-10, 'lambda_2': 2.8279602655612817e-05, 'max_iter': 286, 'tol': 0.0010968160071670654}. Best is trial 0 with value: 0.005820631980895996.
[I 2024-11-20 21:03:08,811] Trial 2 finished with value: 0.005819201469421387 and parameters: {'alpha_1': 1.4057402778539213e-07, 'alpha_2': 3.3392874007037023, 'lambda_1': 0.0442443619097132, 'lambda_2': 0.007840535079786

Best trial:
  Value:  0.005821526050567627
  Params: 
    alpha_1: 1.1773733438411085
    alpha_2: 0.00020568060159971102
    lambda_1: 9.092392261855577
    lambda_2: 4.203760308887446e-07
    max_iter: 118
    tol: 1.3710260703805878e-05


[I 2024-11-20 21:44:46,096] A new study created in memory with name: no-name-8cb22ccb-3834-4e75-9d38-4599a36ffc77


Fold: 2


[I 2024-11-20 21:45:04,010] Trial 0 finished with value: 0.00608748197555542 and parameters: {'alpha_1': 0.00024002752091723776, 'alpha_2': 0.0034665269796631815, 'lambda_1': 3.1942298753594356, 'lambda_2': 0.0007101056942827627, 'max_iter': 592, 'tol': 0.042782583081886254}. Best is trial 0 with value: 0.00608748197555542.
[I 2024-11-20 21:45:22,048] Trial 1 finished with value: 0.006087839603424072 and parameters: {'alpha_1': 0.14532996352588828, 'alpha_2': 0.00028786995534988926, 'lambda_1': 9.316179676389308e-06, 'lambda_2': 3.272734523856613e-06, 'max_iter': 810, 'tol': 0.00016940447626710053}. Best is trial 1 with value: 0.006087839603424072.
[I 2024-11-20 21:45:40,006] Trial 2 finished with value: 0.0060866475105285645 and parameters: {'alpha_1': 0.06805721890776463, 'alpha_2': 2.7449658574044616e-07, 'lambda_1': 2.2421894672470852e-07, 'lambda_2': 0.0012322224861884435, 'max_iter': 333, 'tol': 6.689474979275806e-05}. Best is trial 1 with value: 0.006087839603424072.
[I 2024-11-

Best trial:
  Value:  0.0060898661613464355
  Params: 
    alpha_1: 0.036952272674084116
    alpha_2: 1.3364511242356653e-06
    lambda_1: 9.664985837693665
    lambda_2: 4.1933613057394353e-07
    max_iter: 821
    tol: 0.0007202062188943022


In [8]:
scores = [0.002953171730041504, 0.005821526050567627, 0.0060898661613464355]
sum_scores = sum(scores)
model_weights = [score / sum_scores for score in scores]
model_weights

[0.1986719382804167, 0.3916378625905223, 0.40969019912906096]

In [5]:
# params = study.best_params

# model = BayesianRidge(**params)
# model.fit(train_X,train_y, sample_weight=train_weight)
# y_pred = model.predict(test_X)
# score = custom_metric(test_y,y_pred, test_weight)
# print(f"Score: {score}")

In [6]:
# import joblib
# for fold in range(len(splits)):
#     print(f"Fold: {fold}")
#     if fold == 0:
#         train_index = splits[fold]["train_index"]
#         test_index = splits[fold]["test_index"]
#         X_train, X_test = X[:train_index], X[train_index:test_index]
#         y_train, y_test = y[:train_index], y[train_index:test_index]
#         weight_train, weight_test = weights[:train_index], weights[train_index:test_index]
#     else:
#         prev_train_index = splits[fold - 1]["train_index"]
#         train_index = splits[fold]["train_index"]
#         test_index = splits[fold]["test_index"]
#         X_train, X_test = X[prev_train_index:train_index], X[train_index:test_index]
#         y_train, y_test = y[prev_train_index:train_index], y[train_index:test_index]
#         weight_train, weight_test = weights[prev_train_index:train_index], weights[train_index:test_index]
#     params = best_params[fold]
#     best_model = BayesianRidge(**params)
#     best_model.fit(X_train, y_train, sample_weight=weight_train)
#     joblib.dump(best_model, f"model_{fold}.pkl")

In [7]:
# save model
# import joblib
# joblib.dump(model, "model_2.pkl")